# Portfolio risk report

Market risk for an equity portfolio: Value at Risk, Expected Shortfall,
correlation structure, stress scenarios and a backtest that checks whether
any of it holds up.

Every number below is computed from the modules in `src/`; nothing is
hard-coded. Change the inputs in the next cell and re-run.

## Inputs

Everything that changes the answers lives here.

| knob | effect |
|---|---|
| `WEIGHTS` | the portfolio itself — tickers and allocation. Must sum to 1; negatives are short positions |
| `START` / `END` | the estimation sample. A longer one is more stable and slower to notice a regime change |
| `CONFIDENCE` | 0.99 is the Basel default; 0.975 is what FRTB uses for Expected Shortfall |
| `LAMBDA` | EWMA decay. 0.94 is RiskMetrics daily. Lower reacts faster and is noisier |
| `CORRELATION_WINDOW` | how much history each correlation reading averages over |
| `BACKTEST_WINDOW` | how many days of history each VaR forecast is estimated from |

`START` has to reach back past the oldest stress scenario you want tested —
the 2008 crisis cannot be replayed without 2008 prices.

In [ ]:
WEIGHTS = {"AAPL": 0.4, "MSFT": 0.4, "KO": 0.2}

START = "2006-01-01"
END = "2024-12-31"

CONFIDENCE = 0.99
LAMBDA = 0.94
CORRELATION_WINDOW = 60
BACKTEST_WINDOW = 250

In [ ]:
import warnings

import pandas as pd

from src.backtest import rolling_var_backtest
from src.covariance import correlation, ewma_volatility, rolling_correlation
from src.cvar import historical_cvar
from src.data import load_prices
from src.plots import (
    plot_backtest,
    plot_ewma_volatility,
    plot_return_distribution,
    plot_rolling_correlation,
    plot_stress,
    plot_var_comparison,
)
from src.returns import portfolio_returns, simple_returns
from src.stress import stress_table, uniform_shock
from src.var import historical_var, normal_var, var_summary

%matplotlib inline
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## Data

Adjusted daily closes from Yahoo Finance, cached under `data/`. Only dates on
which every holding traded are kept, so the returns are comparable across
assets.

In [ ]:
prices = load_prices(list(WEIGHTS), START, END)
returns = simple_returns(prices)
portfolio = portfolio_returns(returns, WEIGHTS)

print(f"{len(prices)} trading days, {prices.index.min():%Y-%m-%d} to {prices.index.max():%Y-%m-%d}")
prices.tail(3)

## How much can be lost in a day

Four estimators of the same quantity. They disagree, and the spread is the
point: it separates what the data says from what the assumed distribution
says.

Read the Expected Shortfall column too. VaR is a threshold; ES is the average
loss once that threshold is passed, so it is the one that notices whether the
tail is merely long or genuinely heavy.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    summary = var_summary(portfolio, confidence=CONFIDENCE, seed=1)

summary

In [ ]:
plot_var_comparison(summary);

In [ ]:
plot_return_distribution(
    portfolio,
    historical_var(portfolio, CONFIDENCE),
    historical_cvar(portfolio, CONFIDENCE),
    CONFIDENCE,
);

## What the diversification is worth

A portfolio's risk is whatever survives after its holdings partly cancel each
other out. The uncomfortable part is that the cancelling weakens exactly when
it is needed: correlations rise in a sell-off, so the benefit measured over a
calm sample is not the benefit available in a crisis.

In [ ]:
correlation(returns).round(3)

In [ ]:
plot_rolling_correlation(
    rolling_correlation(returns, window=CORRELATION_WINDOW), CORRELATION_WINDOW
);

In [ ]:
plot_ewma_volatility(ewma_volatility(portfolio, lam=LAMBDA, annualise=True));

## What specific disasters cost

VaR is estimated from a sample that by definition contains nothing worse than
what is already in it. Stress testing asks the other question: never mind the
probability, what does *this* cost?

Historical scenarios replay real windows at today's weights. Nobody has to
defend the assumptions — the market did this. The limit is that the next
crisis will not be a rerun of the last one, which is what the hypothetical
shock below is for.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    stress = stress_table(prices, WEIGHTS)

stress[["portfolio_return", "worst_contributor", "n_days"]]

In [ ]:
plot_stress(stress);

In [ ]:
# A shock nobody has lived through. No probability attached, and none should
# be: the number answers "what would it take", not "what is likely".
stress_table(prices, WEIGHTS, [uniform_shock("Equities -35%", WEIGHTS, -0.35)])[
    ["portfolio_return"]
]

## Does the model actually work

Everything above is a claim. This is the test.

VaR is re-estimated each day from the preceding `BACKTEST_WINDOW` days, so
every comparison is out of sample. At 99% the loss should exceed the forecast
on about 1 day in 100 — and those days should be scattered, not clustered.
Kupiec tests the rate, Christoffersen the timing.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    report = rolling_var_backtest(
        portfolio, historical_var, window=BACKTEST_WINDOW, confidence=CONFIDENCE
    )

print(f"{int(report.breaches.sum())} breaches in {len(report.breaches)} days "
      f"({report.exceedance_rate:.2%}, {1 - CONFIDENCE:.2%} expected)")
print(f"Basel zone: {report.basel_zone}")
report.summary()

In [ ]:
plot_backtest(report);

### Historical against parametric

The experiment worth running: the same returns, the same window, one
assumption changed.

In [ ]:
rows = []
for name, estimator in (("historical", historical_var), ("normal", normal_var)):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        run = rolling_var_backtest(
            portfolio, estimator, window=BACKTEST_WINDOW, confidence=CONFIDENCE
        )
    rows.append(
        {
            "model": name,
            "breaches": int(run.breaches.sum()),
            "rate": run.exceedance_rate,
            "expected": 1 - CONFIDENCE,
            "basel": run.basel_zone,
            "kupiec_p": run.kupiec.p_value,
            "rejected": run.kupiec.rejects(0.01),
        }
    )

pd.DataFrame(rows).set_index("model")

A rejected model is not a failed exercise — it is the exercise working. The
cost of assuming normality is now a number rather than a matter of opinion,
which is the only reason to run a backtest at all.